# neural-playback

**Run your music through a brain.**

Predict neural activation patterns from audio using Meta FAIR's TRIBE v2.

> Scores reflect predicted neural engagement patterns — computational predictions, not clinical measurements.

[![CC BY-NC 4.0](https://img.shields.io/badge/License-CC%20BY--NC%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc/4.0/)
Built by [WESLEYFRANKLIN](https://github.com/wesleyfranklin)

In [ ]:
!pip install -q tribev2 nilearn plotly nibabel soundfile ffmpeg-python
import torch
assert torch.cuda.is_available(), "GPU required. Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")

In [ ]:
import os
if not os.path.exists("neural-playback"):
    !git clone https://github.com/wesleyfranklin/neural-playback.git
%cd neural-playback

## Step 1 — Upload your audio

Supported formats: MP3, WAV, FLAC, M4A

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {audio_path}")

## Step 2 — Preprocess audio

Converts to 16kHz mono MP4 (required by TRIBE v2)

In [ ]:
from neural_playback.preprocess import preprocess_audio
mp4_path = preprocess_audio(audio_path)
print(f"Preprocessed: {mp4_path}")

## Step 3 — Run brain prediction

This may take 1-5 minutes depending on track length.

In [ ]:
from neural_playback.inference import load_model, predict_brain_response
import time

model = load_model()
print("Running brain prediction...")
t0 = time.time()
preds, segments = predict_brain_response(model, mp4_path)
print(f"Done in {time.time()-t0:.1f}s — output shape: {preds.shape}")

## Step 4 — Brain activation render

Regions above activation threshold are labeled with what they do and what it means for your music.

In [ ]:
from neural_playback.visualization import render_brain_annotated
from neural_playback.roi_mapping import load_destrieux_labels, load_roi_map
from IPython.display import Image

labels = load_destrieux_labels()
roi_map = load_roi_map()
png_path = render_brain_annotated(preds, roi_map, labels, timestep=None, threshold=0.5, track_name=audio_path)
Image(str(png_path))

## Step 5 — Activation over time

In [ ]:
from neural_playback.roi_mapping import get_roi_timeseries
from neural_playback.visualization import create_temporal_chart

timeseries = get_roi_timeseries(preds, labels, roi_map)
fig = create_temporal_chart(timeseries, title=f"Neural Activation — {audio_path}")
fig.show()

## Step 6 — Neural Report Card

In [ ]:
from neural_playback.report_card import generate_report_card

scores, report = generate_report_card(preds, labels, track_name=audio_path)
print(report)

## Step 7 — Download results

In [ ]:
from neural_playback.visualization import render_brain_static
import json
from google.colab import files

# Static brain render (2x2 grid, no annotations)
static_path = render_brain_static(preds, output_path="brain_activation.png")

# Report card files
with open("report_card.json", "w") as f:
    json.dump({"track": audio_path, "scores": scores}, f, indent=2)
with open("report_card.txt", "w") as f:
    f.write(report)

files.download("brain_activation.png")
files.download("report_card.json")
files.download("report_card.txt")
print("Downloaded 3 files.")

---

**neural-playback** is open source under [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/).

Built by [WESLEYFRANKLIN](https://github.com/wesleyfranklin) — artist, developer.

Model: [Meta FAIR TRIBE v2](https://github.com/facebookresearch/tribev2)

> Scores reflect predicted neural engagement patterns from Meta FAIR's TRIBE v2 model. These are computational predictions, not clinical measurements.